In [1]:
pip install opencv-python mediapipe


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Dell\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [2]:
import cv2
import mediapipe as mp
import math

In [4]:

# Initialize MediaPipe pose
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Angle calculator
def calculate_angle(a, b, c):
    """Calculate angle between three (x, y) points."""
    x1, y1 = a
    x2, y2 = b
    x3, y3 = c
    angle = math.degrees(
        math.atan2(y3 - y2, x3 - x2) - math.atan2(y1 - y2, x1 - x2)
    )
    return abs((angle + 360) % 360)

# Convert landmark to pixel coordinate
def get_coord(landmarks, idx, w, h):
    lm = landmarks[idx]
    return int(lm.x * w), int(lm.y * h)

# Push-up state variables
stage = None
counter = 0

# Thresholds (tunable)
elbow_min = 80    # at bottom of push-up
elbow_max = 160   # at top of push-up
back_angle_limit = 15  # allowable deviation from straight line

# Start webcam
cap = cv2.VideoCapture(0)

with mp_pose.Pose(min_detection_confidence=0.6, min_tracking_confidence=0.6) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Mirror image for natural feel
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        # Convert to RGB
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        image = frame.copy()

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark

            # Get keypoints
            l_shldr = get_coord(lm, mp_pose.PoseLandmark.LEFT_SHOULDER.value, w, h)
            l_elbow = get_coord(lm, mp_pose.PoseLandmark.LEFT_ELBOW.value, w, h)
            l_wrist = get_coord(lm, mp_pose.PoseLandmark.LEFT_WRIST.value, w, h)
            l_hip = get_coord(lm, mp_pose.PoseLandmark.LEFT_HIP.value, w, h)
            l_knee = get_coord(lm, mp_pose.PoseLandmark.LEFT_KNEE.value, w, h)
            l_ankle = get_coord(lm, mp_pose.PoseLandmark.LEFT_ANKLE.value, w, h)

            # Elbow angle (form check)
            elbow_angle = calculate_angle(l_shldr, l_elbow, l_wrist)
            elbow_color = (0, 255, 0) if elbow_min <= elbow_angle <= elbow_max else (0, 0, 255)

            # Back straightness (shoulder–hip–ankle)
            back_angle = calculate_angle(l_shldr, l_hip, l_ankle)
            back_color = (0, 255, 0) if abs(back_angle - 180) < back_angle_limit else (0, 0, 255)

            # Draw pose
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

            # Visual lines
            cv2.line(image, l_shldr, l_elbow, elbow_color, 3)
            cv2.line(image, l_elbow, l_wrist, elbow_color, 3)
            cv2.line(image, l_shldr, l_hip, back_color, 3)
            cv2.line(image, l_hip, l_ankle, back_color, 3)

            # Display angles
            cv2.putText(image, f"Elbow: {int(elbow_angle)}°", (30, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, elbow_color, 2)
            cv2.putText(image, f"Back: {int(back_angle)}°", (30, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, back_color, 2)

            # Push-up rep counter logic
            if elbow_angle <= elbow_min and abs(back_angle - 180) < back_angle_limit:
                stage = "down"
            if elbow_angle >= elbow_max and stage == "down":
                stage = "up"
                counter += 1

        # Display push-up count
        cv2.putText(image, f"Push-Ups: {counter}", (30, 140),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)

        cv2.imshow("Push-Up Tracker (Press Q to Quit)", image)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
